# RAG Agent 실행 노트북
위에서부터 순서대로 실행하세요. 커널은 프로젝트 가상환경(`.venv`)을 선택해야 합니다.
코드는 같은 폴더의 `ingest.py`, `tools.py`, `agent.py`, `golden.py`에 있습니다.

## 0. 준비

In [ ]:
import os
from pathlib import Path

# Chroma 경로가 현재 폴더 기준이라, 이 노트북이 있는 폴더에서 실행해야 한다.
os.chdir(Path(globals().get("__vsc_ipynb_file__", "run.ipynb")).resolve().parent)
print(os.getcwd())
assert Path("data").exists(), "rag-agent-practice 폴더에서 실행하세요"

import ingest
import tools
import agent

## 1. 문서 처리 (로드 → 청킹 → 임베딩 → Chroma)
`emb_cache/`가 있으면 API를 다시 부르지 않고 Chroma만 새로 만든다. 청킹 설정을 바꾸면(예: 1000/100) 새로 임베딩한다.

In [ ]:
ingest.ingest(chunk_size=500, chunk_overlap=50)

## 2. 검색 도구

In [ ]:
hits = tools.search_documents.invoke({"query": "대장암 증상", "k": 3})
for h in hits:
    print(h["distance"], h["file_name"], f"p{h['page_no']}", h["chunk_id"])
    print("   ", " ".join(h["content"][:100].split()))

## 3. Agent (검색 → 근거 판단 → 재검색 최대 3회 → 답변/보류)

In [ ]:
question = "대장암의 대표적인 증상은 무엇인가요?"
result = agent.run(question)
print(result["status"], result["queries"], f"{result['elapsed_sec']}초")
print(result["answer"])

### 문서에 없는 질문 (보류되는지 확인)

In [ ]:
result = agent.run("토성의 고리는 몇 개이고 무엇으로 이루어져 있나요?")
print(result["status"], result["queries"], f"{result['elapsed_sec']}초")
print(result["answer"])

## 4. 실행 기록 (`runs.jsonl`)

In [ ]:
import pandas as pd

runs = pd.read_json("runs.jsonl", lines=True)
runs[["timestamp", "question", "status", "queries", "elapsed_sec"]]

## 5. 골든셋 (검수 완료 40문항)

In [ ]:
golden = pd.read_csv("golden/golden_eval.csv")
print(golden["domain"].value_counts().to_dict())
golden[["domain", "question", "target_file_name", "target_page_no"]].head(10)

# 평가
검색 단계 비교(청킹, 검색 방식)와 답변 단계 비교(기본 RAG vs Agent)를 같은 골든셋으로 한다. 결과는 `eval_results/`에 캐시되어 다시 실행하면 이어서 진행된다.

**골든셋 3종**
- `verified`: 원문과 대조해 검수한 40문항 (근거 구절 포함)
- `extra`: 원본 `data/golden_set.csv`의 나머지 paragraph·text 문항. **검수하지 않았다** (라벨 노이즈 가능)
- `unanswerable`: 문서에 답이 없는 질문 10개. 정답은 "답변 보류"

**지표**
| 단계 | 지표 | 뜻 |
|---|---|---|
| 검색 | hit@1/3/5, mrr, precision@5 | 정답 페이지의 청크가 상위 k개에 있는가 / 첫 순위 / 상위 5개 중 정답 페이지 비율 |
| 검색 | file_hit@5, evidence_hit@5 | 정답 파일을 찾았는가 / 근거 구절을 담은 청크를 찾았는가 (검수 셋만) |
| 답변 | accuracy | LLM 판정 correct 1, partial 0.5, wrong 0 (보류는 0) |
| 답변 | faithfulness | 답변이 검색된 청크로 뒷받침되는 정도 (답변한 것만) |
| 답변 | token_f1, cite_ok | 기준 답변과 형태소 F1 / 정답 파일·페이지를 출처로 표시한 비율 |
| Agent | first_hit, hit, recovered | 첫 검색 적중 / 최종 적중 / 첫 검색을 놓친 질문을 재검색으로 찾은 비율 |
| 보류 | answered, refusal_rate | 보류하지 않은 비율 / 답 없는 질문을 보류한 비율 |
| 속도 | avg_sec, p95_sec, avg_searches | 응답 시간과 검색 횟수 |

In [ ]:
import evaluate
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
print({d: len(evaluate.load_golden(d)) for d in ["verified", "extra", "unanswerable"]})

## 1. 청킹 비교 (검색 방식은 벡터로 고정)

In [ ]:
CHUNKS = [(500, 50), (1000, 100)]
for size, overlap in CHUNKS:
    ingest.ingest(size, overlap)  # 임베딩 캐시가 있으면 API 호출 없이 Chroma만 만든다

chunk_df = evaluate.compare_retrieval([("vector", s, o) for s, o in CHUNKS])
chunk_df.drop(columns=["method"])

## 2. 검색 방식 비교 (청킹 고정)

In [ ]:
def pooled(df, metric="hit@5"):
    # 검수 셋과 미검수 셋을 문항 수로 가중해 합친 값. 40문항만으로는 차이가 우연일 수 있어서 117문항으로 보강한다.
    return (df[metric] * df["n"]).groupby([df["method"], df["chunk"]]).sum() / df.groupby(["method", "chunk"])["n"].sum()


best_chunk = pooled(chunk_df).idxmax()[1]
CHUNK = tuple(int(x) for x in best_chunk.split("/"))
print("선택한 청킹:", best_chunk)

METHODS = ["vector", "bm25", "hybrid"]
method_df = evaluate.compare_retrieval([(m, *CHUNK) for m in METHODS])
method_df.drop(columns=["chunk"])

In [ ]:
# 참고: 청킹 × 검색 방식 전체 조합 (hit@5, 검수 셋 + 미검수 셋을 합친 값)
grid = evaluate.compare_retrieval([(m, s, o) for m in METHODS for s, o in CHUNKS])
pd.DataFrame({"hit@5": pooled(grid, "hit@5"), "mrr": pooled(grid, "mrr"), "precision@5": pooled(grid, "precision@5")}).round(3)

### 선택
청킹은 위 1번에서 고른 값으로 두고, 검색 방식은 두 골든셋을 합친 hit@5(같으면 mrr)가 가장 높은 것을 고른다.

In [ ]:
score = pd.DataFrame({"hit@5": pooled(method_df, "hit@5"), "mrr": pooled(method_df, "mrr")})
METHOD = score.sort_values(["hit@5", "mrr"], ascending=False).index[0][0]
print("선택한 검색 방식:", METHOD)
tools.SETTINGS.update(chunk_size=CHUNK[0], chunk_overlap=CHUNK[1], method=METHOD)
tools.SETTINGS

## 3. 기본 RAG vs Agent (선택한 설정으로 고정)
기본 RAG는 한 번 검색해서 바로 답하고, Agent는 근거를 판단해 재검색(최대 3회)하거나 보류한다. 기본값(vector, 500/50) 기본 RAG도 함께 돌려 설정 선택이 답변까지 영향을 주는지 본다.

In [ ]:
tag = f"{METHOD}_{CHUNK[0]}"
DEFAULT = dict(chunk_size=500, chunk_overlap=50, method="vector")
CHOSEN = dict(chunk_size=CHUNK[0], chunk_overlap=CHUNK[1], method=METHOD)
runs = {f"basic_{tag}": (evaluate.basic_rag, CHOSEN), f"agent_{tag}": (evaluate.agent_rag, CHOSEN)}
if tag != "vector_500":
    runs["basic_vector_500"] = (evaluate.basic_rag, DEFAULT)  # 기준선: 검색 설정만 다르다

for dataset in ["verified", "extra", "unanswerable"]:
    for name, (fn, cfg) in runs.items():
        tools.SETTINGS.update(cfg)  # run_system은 호출 시점의 tools.SETTINGS로 검색한다
        evaluate.run_system(name, fn, dataset)
tools.SETTINGS.update(CHOSEN)
names = list(runs)
print("done")

### 3-1. 종합 비교

In [ ]:
summary = evaluate.compare_systems(names)
answerable = summary[summary.dataset != "unanswerable"].dropna(axis=1, how="all")
answerable

In [ ]:
# 답이 없는 질문: 보류해야 정답. refusal_rate가 높을수록 지어내지 않는다
summary[summary.dataset == "unanswerable"].dropna(axis=1, how="all")

### 3-2. 도메인별 accuracy (검수 셋 / 미검수 셋)

In [ ]:
print("verified"); display(evaluate.by_domain(names, "verified"))
print("extra"); display(evaluate.by_domain(names, "extra"))

### 3-3. 오답 원인 분류
검색 실패 = 정답 페이지를 못 찾고 답한 경우, 생성 실패 = 정답 페이지를 찾았는데 틀린 경우, 보류 = Agent가 답을 보류한 경우

In [ ]:
for dataset in ["verified", "extra"]:
    print(dataset); display(evaluate.failure_breakdown(names, dataset))

### 3-4. 질문별 승패 (Agent가 기본 RAG보다 나은 질문 수)

In [ ]:
basic, agentic = f"basic_{tag}", f"agent_{tag}"
pd.DataFrame({d: evaluate.win_loss(agentic, basic, d) for d in ["verified", "extra"]}).T

### 3-5. 실패 사례 (Agent, wrong 판정)

In [ ]:
evaluate.show_failures(f"agent_{tag}", "verified", limit=8)

## 결론

아래 수치는 이 노트북을 한 번 실행한 결과다. 골든셋은 검수 40문항(verified), 미검수 117문항(extra), 답 없는 질문 10문항이다. 표본이 작아서 2~3%p 차이는 1~3문항 차이일 뿐이니 조심해서 읽어야 한다.

### 1. 청킹: 500/50 vs 1000/100 (벡터 검색 고정)
| | 셋 | hit@5 | mrr | precision@5 |
|---|---|---|---|---|
| 500/50 | verified / extra | 0.950 / 0.889 | 0.797 / 0.795 | 0.340 / 0.330 |
| 1000/100 | verified / extra | 0.975 / 0.897 | 0.747 / 0.789 | 0.250 / 0.234 |

- hit@5는 1000/100이 근소하게 높아서(합산 0.917 vs 0.905) 선택했지만, 이 차이는 **1~2문항 수준이라 유의하다고 보기 어렵다.**
- 큰 청크는 같은 k에서 페이지 텍스트를 더 많이 담으므로 hit@5가 기계적으로 유리하다. 반대로 mrr과 precision@5는 500/50이 높다. 즉 작은 청크는 정답 위치를 더 위로 올리고, 큰 청크는 정답 페이지를 놓칠 확률을 줄인다.
- 구분해서 말할 수 있는 건 "1000/100이 확실히 낫다"가 아니라 "둘이 비슷하고 지표마다 유리한 쪽이 다르다"까지다.

### 2. 검색 방식: 벡터 vs BM25 vs Hybrid (1000/100 고정)
| | 셋 | hit@1 | hit@5 | mrr |
|---|---|---|---|---|
| vector | verified / extra | 0.600 / 0.709 | 0.975 / 0.897 | 0.747 / 0.789 |
| bm25 | verified / extra | 0.700 / 0.769 | 1.000 / 0.949 | 0.810 / 0.845 |
| hybrid | verified / extra | 0.800 / 0.795 | 1.000 / 0.915 | 0.875 / 0.850 |

- 선택 규칙(두 셋 합산 hit@5, 같으면 mrr)으로는 **BM25**가 뽑혔다(0.962, 하이브리드 0.937, 벡터 0.917). 벡터와 BM25의 차이는 157문항 중 약 7문항이라 청킹 차이보다는 믿을 만하다.
- 다만 **순위 품질(hit@1, mrr)은 Hybrid가 가장 좋다.** hit@5 기준이라 BM25가 뽑혔을 뿐, Agent에 쓰는 k가 5이므로 이 선택은 합리적이지만 Hybrid도 대안이다.
- 주의: 골든셋 질문이 원문 용어를 그대로 쓰는 경우가 많아 **어휘 일치 기반인 BM25에 유리할 수 있다.** 사용자가 바꿔 말한 질문을 던지면 벡터가 더 나을 수 있는데, 이 골든셋으로는 확인하지 못했다.
- `avg_ms`는 비교하지 말 것. 질문 임베딩을 캐시하는 구조라 처음 평가한 설정만 임베딩 시간이 포함된다. 속도는 아래 `avg_sec`(종단 간)로 본다.

### 3. 기본 RAG vs Agent (bm25, 1000/100)
| 시스템 | 셋 | accuracy | answered | cite_ok | avg_sec | 검색 횟수 |
|---|---|---|---|---|---|---|
| 기본 RAG | verified / extra | 0.950 / 0.910 | 1.00 / 1.00 | 0.975 / 0.932 | 6.3 / 7.0 | 1.0 |
| Agent | verified / extra | 0.938 / 0.829 | 0.975 / 0.855 | 0.923 / 0.990 | 13.0 / 13.0 | 1.05 / 1.32 |
| 기본 RAG (vector, 500/50) | verified / extra | 0.900 / 0.889 | 1.00 / 1.00 | 0.900 / 0.889 | 6.4 / 7.1 | 1.0 |

- **이번 설정에서 Agent는 기본 RAG보다 나빴다.** verified에서는 같거나 1문항 차이(나은 질문 0, 못한 질문 1), extra에서는 나은 질문 4, 못한 질문 13이다. 응답 시간은 약 2배다.
- 원인은 **보류가 너무 많아서**다. extra에서 Agent가 보류한 17문항 중 **12문항은 정답 페이지를 이미 검색해 놓고도 보류**했고, 같은 17문항을 기본 RAG는 10문항 맞혔다. 질문이 여러 가지를 한꺼번에 묻는 경우(예: "기금형과 계약형의 차이와 장단점") 근거 판단이 "일부만 있으니 부족"이라고 판정하는 것으로 보인다.
- 재검색의 효과는 작다. 첫 검색을 놓친 질문이 6개뿐인데 그중 재검색으로 찾은 건 1개(recovered 0.167)다. 첫 검색 적중률이 이미 95%라 재검색이 개입할 여지가 거의 없다.
- 오답의 대부분은 검색 실패가 아니라 **생성 실패**(찾았는데 틀림 또는 일부만 맞음)다. 기본 RAG는 extra에서 틀린 16개 중 6개만 검색 실패였다.
- 답 없는 질문 10개는 **기본 RAG도 전부 "찾지 못했다"고 답했다**(refusal 1.0). 프롬프트의 "결과에 없는 내용은 추측하지 마라"만으로 충분했다는 뜻이다. 그래서 이 골든셋에서는 Agent의 근거 판단과 보류 장치가 얻은 이득이 없고 비용(시간 약 5배, 검색 3회)만 들었다. 질문이 10개뿐이라 일반화는 조심해야 한다.
- 설정 선택의 효과: 같은 기본 RAG에서 vector/500 대비 bm25/1000이 accuracy가 verified 0.900→0.950, extra 0.889→0.910으로 올랐다. 검색 개선은 답변에도 전달됐다.

### 최종 설정과 이유
**BM25 + 청킹 1000/100, 기본 RAG를 우선 사용하고 Agent는 보류 기준을 고친 뒤 다시 평가한다.** 검색 방식은 합산 hit@5가 가장 높았고 답변 accuracy에도 이어졌다. Agent는 지금 상태로는 정확도와 속도가 모두 기본 RAG보다 나빠서 선택하지 않았다.

### 실패 사례와 개선점
1. **Agent의 과잉 보류**: 근거 판단 프롬프트가 "구체적인 내용이 없으면 부족"이라 엄격하다. 일부라도 있으면 답하고 부족한 부분을 명시하도록 바꾸거나, 보류는 관련 청크가 전혀 없을 때만 하도록 조정한다. 이후 extra에서 보류 17건이 줄어드는지로 확인한다.
2. **생성 실패**: 질문이 여러 부분을 묻는데 답이 일부만 담는다. k를 늘리거나 답변 프롬프트에 "질문의 모든 부분에 답하라"를 넣어 본다.
3. **평가의 한계**: (a) faithfulness가 모든 답변에서 yes라 변별력이 없다(판정이 관대하거나 답이 실제로 충실한 것이다). 더 엄격한 문장 단위 검증이 필요하다. (b) extra는 검수하지 않아 라벨이 틀렸을 수 있다(앞서 정답 페이지 일치율 약 89%). (c) table·image 문항은 텍스트 추출로는 답할 수 없어 제외했다. (d) 표본이 작아 청킹 차이는 확정할 수 없다.
4. **검색 평가 편향**: 원문 용어를 쓴 질문이 많아 BM25에 유리하다. 바꿔 말한 질문 세트를 추가해 확인한다.

# 표 추출 비교
`pdfplumber`로 표를 찾아 Markdown 청크로 덧붙인 인덱스(`_tbl`)와 기존 텍스트 인덱스를 비교한다. 설정은 앞에서 고른 BM25 + 1000/100으로 고정한다.

- `table`: 원본 골든셋에서 표를 보고 답하는 48문항. **검수하지 않았다.**
- 인덱스를 처음 만들 때는 모든 페이지에서 표를 찾느라 20분 안팎 걸린다 (`emb_cache/pdf_c1000_o100_tbl.pkl`이 있으면 바로 끝난다).

In [ ]:
ingest.ingest(1000, 100, tables=True)  # 본문은 그대로 + 표 Markdown 청크를 덧붙인다

In [ ]:
# 표 청크가 실제로 어떻게 생겼는지
store = tools.vectorstore(1000, 100, True)
table_chunk = store._collection.get(where={"content_type": "table"}, limit=1, include=["documents", "metadatas"])
print(table_chunk["metadatas"][0]["file_name"], "p", table_chunk["metadatas"][0]["page_no"])
print(table_chunk["documents"][0][:500])

## 검색 비교 (텍스트 전용 vs 표 추출)

In [ ]:
configs = [(m, 1000, 100, t) for m in ["vector", "bm25", "hybrid"] for t in (False, True)]
tbl_df = evaluate.compare_retrieval(configs, datasets=("table", "verified", "extra"))
tbl_df[["dataset", "method", "tables", "n", "hit@1", "hit@5", "mrr", "precision@5", "file_hit@5"]]

## 답변 비교 (기본 RAG, BM25 + 1000/100)

In [ ]:
base = dict(chunk_size=1000, chunk_overlap=100, method="bm25")
variants = {"basic_bm25_1000": dict(base, tables=False), "basic_bm25_1000_tbl": dict(base, tables=True)}
for dataset in ["table", "verified", "extra"]:
    for name, cfg in variants.items():
        tools.SETTINGS.update(cfg)
        evaluate.run_system(name, evaluate.basic_rag, dataset)
tools.SETTINGS.update(CHOSEN)  # 이후 셀을 위해 원래 설정으로
names_tbl = list(variants)
evaluate.compare_systems(names_tbl, ["table", "verified", "extra"]).dropna(axis=1, how="all")

In [ ]:
# 질문별 승패 (표 추출이 나은 / 같은 / 못한 질문 수)와 오답 원인
print(pd.DataFrame({d: evaluate.win_loss("basic_bm25_1000_tbl", "basic_bm25_1000", d) for d in ["table", "verified", "extra"]}).T)
evaluate.failure_breakdown(names_tbl, "table")

## 결론: 표 추출
| 시스템 (기본 RAG, bm25 1000/100) | 표 문항 accuracy | 검수 40문항 | 미검수 117문항 |
|---|---|---|---|
| 텍스트 전용 | 0.823 | 0.950 | 0.910 |
| 표를 본문에서 떼어내는 1차 시도 (`tblcut`) | **0.510** | 0.975 | 0.932 |
| 본문은 그대로 + 표 청크 덧붙임 (최종) | **0.865** | 0.950 | 0.927 |

**1차 시도는 실패했다.** pdfplumber가 표를 감지하고도 병합 셀·세로 글자 칸을 빈칸으로 추출했는데, 표 영역의 글자를 본문에서 빼서 그 글자가 어디에도 남지 않았다. 정답 페이지 청크를 열어 보면 법령 표에서 날짜·법률번호 열이, 지급비 표에서 행 이름(의료비 등)이 비어 있었다. 그 결과 정답 페이지를 찾고도 틀리는 생성 실패가 3개에서 19개로 늘었다.

**최종 방식(덧붙임)의 결과**
- 표 문항에서 accuracy 0.823 → 0.865, 질문별로는 나아진 질문 3, 같은 질문 45, 못한 질문 0이다. 다만 표본이 48문항이라 **3개 차이는 우연일 수 있다.**
- 검색은 방식마다 다르다. 표 문항 hit@5는 벡터 0.896 → 0.938, BM25 0.875 → 0.896이고 Hybrid는 0.896 → 0.875로 오히려 1문항 줄었다. hit@1과 mrr은 세 방식 모두 올랐다.
- precision@5가 크게 오른 것(예: BM25 0.196 → 0.325)은 같은 페이지에서 청크가 더 많아졌기 때문이라 **기계적으로 부풀려진 값**이다. 검색이 좋아졌다는 근거로 쓰면 안 된다.
- 기존 문항에서는 나빠지지 않았다. 검수 40문항은 변화가 없고, 미검수 117문항은 나아진 질문 5, 못한 질문 1이다.
- 개선 폭이 작은 이유: 표 문항 정답 숫자의 89%는 원래 PyPDF 본문에도 들어 있었다(표를 평평한 텍스트로 읽어 온 것). 표 구조를 살려도 LLM이 풀 수 있던 문항이 대부분이라 새로 얻는 건 일부다.

**남은 한계**
- 글자가 이미지인 표·차트는 여전히 못 읽는다. Gemini 비전으로 읽는 처리는 넣지 않았다. 표 문항에서 아직 틀리는 7개 중 이런 경우가 얼마나 되는지는 확인하지 않았다.
- 병합 셀이 있는 표는 여전히 빈칸이 생긴다. 본문을 남겨 둔 덕분에 정보는 잃지 않지만, 표 청크 자체의 품질은 낮다.
- 표 청크가 본문과 내용이 겹쳐서 같은 내용이 중복으로 검색될 수 있다.